# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jawad-ahmed-developer/flyRank_Internship_Tasks/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building the approved feature vector

ML-04 already defined the data contract and selected five features for the Refresh / Content Opportunity Scoring lane. This notebook does not re-select those features. Instead, it turns that contract into the actual model input:

**imp_prev30**— impressions from the previous 30-day window.

**clicks_prev30** — clicks from the previous 30-day window.

**avg_position_prev30**— average search position from the previous 30-day window.

**content_age_days** — age of the content at the decision moment.

**days_since_last_update** — time since the content was last updated at the decision moment.

The first three describe historical Search Console performance, while the last two describe the content state at the same decision point. The feature vector therefore contains information available before the future outcome is evaluated.

IDs are retained only outside X so that pages can be identified or grouped; they are not model features.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

%pip -q install duckdb huggingface_hub pandas numpy scikit-learn

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load Hugging Face token securely.
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face READ token (hf_...): "
    )

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not provided.")

print("Hugging Face token loaded.")

Enter your Hugging Face READ token (hf_...): ··········
Hugging Face token loaded.


In [3]:
con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_content": (
        f"read_parquet('{REL}/dim_content.parquet')"
    ),
    "fact_daily": (
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
    )
}

print("Warehouse connection established.")

Warehouse connection established.


In [5]:
# Decision point: after March 30, 2026.
# Therefore, the previous 30 days are March 1 through March 30.
DECISION_MONTH = "2026-03"

FEATURES = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "content_age_days",
    "days_since_last_update",
]

historical = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        -- Total Search Console impressions during the previous 30 days
        SUM(COALESCE(gsc_impressions, 0)) AS imp_prev30,

        -- Total Search Console clicks during the previous 30 days
        SUM(COALESCE(gsc_clicks, 0)) AS clicks_prev30,

        -- Average Search Console position during the previous 30 days.
        -- 0 means no position data, so it is excluded from the average.
        AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30

    FROM {TABLES["fact_daily"]}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date <  DATE '2026-03-31'

      -- Only use rows where GSC data was actually available.
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print("Historical feature rows:", len(historical))
display(historical.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Historical feature rows: 175205


,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,76.0,0.0,4.725188
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10305.0,22.0,8.293522
2,client_62f4a7e64f5e0096,content_e689bc511192751a,59.0,0.0,7.018939
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,692.0,1.0,6.083861
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,14.343567


In [6]:
content_meta = con.sql(
    f"""
    SELECT
        content_hash_id,
        content_created_at,
        last_updated_at
    FROM {TABLES["dim_content"]}
    """
).df()

decision_date = pd.Timestamp("2026-03-30")

content_meta["content_age_days"] = (
    decision_date - pd.to_datetime(content_meta["content_created_at"])
).dt.days

content_meta["days_since_last_update"] = (
    decision_date - pd.to_datetime(content_meta["last_updated_at"])
).dt.days

feature_frame = historical.merge(
    content_meta[
        [
            "content_hash_id",
            "content_age_days",
            "days_since_last_update",
        ]
    ],
    on="content_hash_id",
    how="left"
)

print("Feature frame shape:", feature_frame.shape)

display(
    feature_frame[
        ["content_hash_id"] + FEATURES
    ].head()
)

BinderException: Binder Error: Referenced column "content_created_at" not found in FROM clause!
Candidate bindings: "content_created_date", "content_updated_date", "content_type", "content_hash_id", "keyword_created_date"

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.